In [ ]:
#a. Quitar tildes de los campos de texto.
def quitar_tildes(texto):
    if pd.isna(texto):
        return texto
    return "".join(c for c in unicodedata.normalize("NFD", texto) if unicodedata.category(c) != "Mn")

In [ ]:
#b. Función de mapeo de géneros de libros.
def mapear_genero_libro(genero):
    for clave, valores in map_genero.items():
        if genero in valores:
            return clave
    return genero  # si no matchea nada, lo dejo sin agrupar (para detectarlo)

In [ ]:
#c. Función para calcular rating promedio dejando afuera el registro actual (leave-one-out, evita leakage).
def loo_mean(df, group_cols, target_col="rating"):
    grp = df.groupby(group_cols)[target_col]
    return (grp.transform("sum") - df[target_col]) / (grp.transform("count") - 1)

In [ ]:
#d. Replica del feature engineering sobre dataframe a predecir.
def feature_engineering_test(id_lector, id_libros):
    
    #1. Creo las combinaciones lector-libro candidato.
    #print("Creo las combinaciones id_lector-id_libro.")
    df_test_features = pd.DataFrame({
        "id_lector": id_lector,
        "id_libro": id_libros
    })
    
    #2. Mergeo características del lector.
    #print("Mergeo características del lector.")
    df_test_features = df_test_features.merge(
        caract_lector,
        on="id_lector",
        how="left"
    )
    
    #3. Mergeo características del libro.
    #print("Mergeo características del libro.")
    df_test_features = df_test_features.merge(
        caract_libros,
        on="id_libro",
        how="left"
    )

    #4. Mergeo afinidad lector-autor.
    #print("Mergeo características de afinidad lector-autor.")
    df_test_features = df_test_features.merge(
        afinidad_lector_autor_test,
        on=["id_lector", "autor"],
        how="left"
    )
    
    #5. Completo variables de afinidad lector-autor.
    
    # Si nunca interactuó con el autor, cantidad = 0.
    df_test_features["n_interacciones_lector_autor"] = (
        df_test_features["n_interacciones_lector_autor"]
        .fillna(0)
    )
    
    # Si nunca interactuó con el autor, rating = media global.
    df_test_features["rating_prom_id_lector_autor"] = (
        df_test_features["rating_prom_id_lector_autor"]
        .fillna(media_global)
    )

    #6. Mergeo afinidad lector-género.
    #print("Mergeo características de afinidad lector-género.")
    df_test_features = df_test_features.merge(
        afinidad_lector_genero_test,
        on=["id_lector", "genero_libro_agrupado"],
        how="left"
    )
    
    #7. Completo variables de afinidad lector-género.
    
    # Si nunca interactuó con el género, cantidad = 0.
    df_test_features["n_interacciones_lector_genero"] = (
        df_test_features["n_interacciones_lector_genero"]
        .fillna(0)
    )
    
    # Si nunca interactuó con el género, rating = media global.
    df_test_features[
        "rating_prom_id_lector_genero_libro_agrupado"
    ] = (
        df_test_features[
            "rating_prom_id_lector_genero_libro_agrupado"
        ]
        .fillna(media_global)
    )
    
    #8. Features temporales para candidatos.
    #print("Calculo características temporales de la interacción.")
    
    df_test_features["edad_al_interactuar"] = (
        anio_actual - df_test_features["nacimiento"]
    )

    # No existe una interacción real todavía.
    #df_test_features["dias_transcurridos_interaccion"] = 0 # sacar ya que en test todo va a ser 0.

    df_test_features["anios_transcurridos_edicion"] = (
        anio_actual - df_test_features["anio_edicion"]
    )

    #df_test_features["antiguedad_libro_hoy"] = ( 
    #    anio_actual - df_test_features["anio_edicion"] # sacar ya que en test todo va a ser a anios_transcurridos_edicion.
    #)
    
    return df_test_features

In [ ]:
#e. Me devuelve por id_lector todos los id_libros que son candidatos a ser recomendados.
def retrieval(id_lector):
    """Retorna todos los libros que se pueden recomendar a id_lector."""
    
    libros_leidos = leidos_por_lector.get(id_lector, set())
    libros_no_leidos = list(set(todos_los_libros) - libros_leidos)
    
    return libros_no_leidos

In [ ]:
#f. Devuelvo un score predicho para cada id_libro candidato de un id_lector.
def ranking(df_features, features_modelo, modelo):
    """Predice el rating de cada libro candidato a partir de sus features ya calculadas."""

    #1. Selecciono exactamente las variables que utiliza el modelo.
    X = df_features[features_modelo]

    #2. Predigo el rating para cada libro candidato.
    predicciones = modelo.predict(X)

    #3. Devuelvo un diccionario {id_libro: rating_predicho}.
    return dict(zip(df_features["id_libro"], predicciones))

In [ ]:
#g. Función para limpiar el año de edición.
def limpiar_anio_edicion(df, anio_actual):
    """
    Limpia y completa la columna 'anio_edicion'.

    Pasos:
    1. Extrae años de 4 dígitos.
    2. Convierte valores inválidos a NaN.
    3. Elimina años fuera del rango [1800, anio_actual].
    4. Imputa faltantes con el promedio del título.
    5. Imputa los restantes con el promedio de la editorial.
    6. Imputa los restantes con la mediana general.
    7. Convierte la columna a int.
    """

    #1. Extraer año válido y convertir a numérico
    df["anio_edicion"] = (
        df["anio_edicion"]
        .astype(str)
        .str.extract(r"(\d{4})")[0]
    )

    df["anio_edicion"] = pd.to_numeric(
        df["anio_edicion"],
        errors="coerce"
    )

    print(
        f"Nulos tras extraer año válido: "
        f"{df['anio_edicion'].isnull().sum()}"
    )

    #2. Nulificar años fuera de rango
    mask_fuera_rango = (
        (df["anio_edicion"] < 1800) |
        (df["anio_edicion"] > anio_actual)
    )

    print(
        f"Años fuera de rango [1800, {anio_actual}] "
        f"nulificados: {mask_fuera_rango.sum()}"
    )

    df.loc[mask_fuera_rango, "anio_edicion"] = np.nan

    #3. Imputar por promedio del título
    promedio_por_titulo = (
        df.groupby("titulo")["anio_edicion"]
        .transform("mean")
        .round()
    )

    df["anio_edicion"] = df["anio_edicion"].fillna(
        promedio_por_titulo
    )

    print(
        f"Nulos tras imputar por promedio de título: "
        f"{df['anio_edicion'].isnull().sum()}"
    )

    #4. Imputar por promedio de editorial
    promedio_por_editorial = (
        df.groupby("editorial")["anio_edicion"]
        .transform("mean")
        .round()
    )

    df["anio_edicion"] = df["anio_edicion"].fillna(
        promedio_por_editorial
    )

    print(
        f"Nulos tras imputar por promedio de editorial: "
        f"{df['anio_edicion'].isnull().sum()}"
    )

    #5. Imputar restantes con mediana general
    mediana_general = round(df["anio_edicion"].median())

    df["anio_edicion"] = df["anio_edicion"].fillna(
        mediana_general
    )

    print(
        f"Nulos tras imputar por mediana general: "
        f"{df['anio_edicion'].isnull().sum()}"
    )

    #6. Convertir a entero
    df["anio_edicion"] = df["anio_edicion"].astype(int)

    return df

In [ ]:
#h. Limpiar la fecha de nacimiento.
def limpiar_nacimiento(df, anio_actual):
    """
    Limpia y completa la columna 'nacimiento'.

    Pasos:
    1. Convierte la columna a numérico.
    2. Nulifica años fuera del rango [1900, anio_actual].
    3. Imputa faltantes con el promedio de nacimiento por nombre.
    4. Imputa los restantes con el promedio global.
    5. Convierte la columna a entero.
    """

    #1. Forzar a numérico
    df["nacimiento"] = pd.to_numeric(
        df["nacimiento"],
        errors="coerce"
    )

    print(
        f"Nulos tras forzar a numérico: "
        f"{df['nacimiento'].isnull().sum()}"
    )

    #2. Nulificar nacimientos fuera de rango
    mask_fuera_rango = (
        (df["nacimiento"] < 1900) |
        (df["nacimiento"] > anio_actual)
    )

    print(
        f"Nacimientos fuera de rango [1900, {anio_actual}] "
        f"nulificados: {mask_fuera_rango.sum()}"
    )

    df.loc[mask_fuera_rango, "nacimiento"] = np.nan

    #3. Imputar por promedio del nombre
    promedio_por_nombre = (
        df.groupby("nombre")["nacimiento"]
        .transform("mean")
        .round()
    )

    df["nacimiento"] = df["nacimiento"].fillna(
        promedio_por_nombre
    )

    print(
        f"Nulos tras imputar por promedio de nombre: "
        f"{df['nacimiento'].isnull().sum()}"
    )

    #4. Imputar restantes con promedio global
    promedio_global = round(df["nacimiento"].mean())

    df["nacimiento"] = df["nacimiento"].fillna(
        promedio_global
    )

    print(
        f"Nulos tras imputar por promedio global: "
        f"{df['nacimiento'].isnull().sum()}"
    )

    #5. Chequeo final
    print(
        f"Nulos restantes: "
        f"{df['nacimiento'].isnull().sum()}"
    )

    #6. Convertir a entero
    df["nacimiento"] = df["nacimiento"].astype(int)

    return df